In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

In [12]:
# 1. Load and Prepare Data
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# Feature Engineering
for df in [train_df, test_df]:
    df['u_minus_g'] = df['u'] - df['g']
    df['g_minus_r'] = df['g'] - df['r']
    df['r_minus_i'] = df['r'] - df['i']
    df['i_minus_z'] = df['i'] - df['z']
    df['total_mag'] = df['u'] + df['g'] + df['r'] + df['i'] + df['z']

X_train = train_df.drop(['id', 'class'], axis=1)
y_train = train_df['class']
X_test = test_df.drop(['id'], axis=1)

# Encode Categoricals
cat_cols = ['spectral_type', 'galaxy_population']
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])

# Encode Target Variable (Crucial for XGBoost)
target_le = LabelEncoder()
y_train_encoded = target_le.fit_transform(y_train) # Converts to 0, 1, 2

In [13]:
# 2. Define Models
models = {
    'lgb': lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, random_state=42, verbose=-1, n_jobs=-1),
    'xgb': xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, random_state=42, verbosity=0, n_jobs=-1, tree_method='hist'),
    'rf': RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced', n_jobs=-1)
}

In [14]:
# 3. Train and Predict
print("--- Training Ensemble Models ---")
all_preds_encoded = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train_encoded) # Use encoded targets
    preds_encoded = model.predict(X_test)
    all_preds_encoded.append(preds_encoded)


--- Training Ensemble Models ---
Training lgb...
Training xgb...
Training rf...


In [15]:
# 4. Combine Predictions (Majority Vote on Encoded Values)
preds_df = pd.DataFrame(np.array(all_preds_encoded).T, columns=['lgb', 'xgb', 'rf'])
ensemble_preds_encoded = preds_df.mode(axis=1)[0].values

In [17]:
# 5. Convert Back to String Labels for Submission
final_predictions = target_le.inverse_transform(ensemble_preds_encoded.astype(int))

In [18]:
# 6. Create Submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'class': final_predictions
})

submission.to_csv('submission_v4_ensemble.csv', index=False)
print("Success! submission_v4_ensemble.csv is ready.")
print("\nFirst few rows:")
print(submission.head())

Success! submission_v4_ensemble.csv is ready.

First few rows:
       id   class
0  577347  GALAXY
1  577348  GALAXY
2  577349  GALAXY
3  577350    STAR
4  577351  GALAXY
